# MNIST Problem

## Imports

In [1]:
import numpy as np
import tensorflow as tf

import tensorflow_datasets as tfds

I0000 00:00:1781716888.293852   13157 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781716888.597226   13157 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1781716892.034202   13157 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## Data

In [2]:
# Fetch data from the tensorflow datasets

mnist_dataset,mnist_info = tfds.load(name='MNIST',with_info=True,as_supervised=True)

I0000 00:00:1781716896.480139   13157 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3537 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


## Train-Validation-Test

In [3]:
# Splitting the data in train and test
mnist_train,mnist_test = mnist_dataset['train'],mnist_dataset['test']

# Set size of validation data - 10%
validation_size = 0.1*mnist_info.splits['train'].num_examples
validation_size = tf.cast(validation_size,tf.int64)

# Set test data size
test_size = mnist_info.splits['test'].num_examples
test_size = tf.cast(test_size,tf.int64)

## Scaling data

In [4]:
# Defining scale function for map

def scale(image,label):
    image = tf.cast(image,tf.float32)
    image /= 255. # divides each pixel by 255 and returns a float

    return image,label

# Scaling train,validation and test data

scaled_train_validation_data = mnist_train.map(scale)
scaled_test_data = mnist_test.map(scale)


## Preprocessing

In [5]:
Buffer_size = 10000  # To take limited number at a time

# Shuffle data
shuffle_train_val_data = scaled_train_validation_data.shuffle(Buffer_size)

validation_data = shuffle_train_val_data.take(validation_size)
train_data = shuffle_train_val_data.skip(validation_size)

test_data = scaled_test_data

# Creating Batches

Batch_size = 100

train_data = train_data.batch(Batch_size)
validation_data = validation_data.batch(validation_size)
test_data = test_data.batch(test_size)

validation_input,validation_output = next(iter(validation_data))

I0000 00:00:1781716900.566935   13251 tf_record_dataset_op.cc:396] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608
W0000 00:00:1781716901.225355   13255 cache_dataset_ops.cc:912] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


## Model

### Outline Model

In [6]:
# Layers Size

input_size = 728
output_size = 10
hidden_layer_size = 50


# Model

model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28,28,1)),  # Input Layer
    tf.keras.layers.Dense(hidden_layer_size,activation='relu'), # Hidden Layer 1
    tf.keras.layers.Dense(hidden_layer_size,activation='relu'), # Hidden Layer 2
    tf.keras.layers.Dense(hidden_layer_size,activation='relu'), # Hidden Layer 3
    tf.keras.layers.Dense(output_size,activation='softmax') # Output Layer
])

/home/pranav/miniconda3/envs/tf/lib/python3.11/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


### Optimizer and Loss Function

In [7]:
model.compile(optimizer='adam',loss = 'sparse_categorical_crossentropy',metrics=['accuracy'])

## Training

In [8]:
NO_OF_EPOCHS = 15

early_stopping = tf.keras.callbacks.EarlyStopping(patience=3)

model.fit(train_data,
          callbacks=[early_stopping],
          epochs=NO_OF_EPOCHS,
          validation_data=(validation_input,validation_output),
          verbose=2)

Epoch 1/15


I0000 00:00:1781716904.656629   13232 service.cc:153] XLA service 0x7ddf70033100 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1781716904.656661   13232 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4050 Laptop GPU, Compute Capability 8.9 (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.23.1)
I0000 00:00:1781716904.680090   13232 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1781716904.885805   13232 cuda_dnn.cc:461] Loaded cuDNN version 92301
I0000 00:00:1781716904.894483   13232 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1680__.14
I0000 00:00:1781716904.919012   13232 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set i

540/540 - 19s - 35ms/step - accuracy: 0.8767 - loss: 0.4070 - val_accuracy: 0.9418 - val_loss: 0.2038
Epoch 2/15
540/540 - 5s - 10ms/step - accuracy: 0.9510 - loss: 0.1641 - val_accuracy: 0.9597 - val_loss: 0.1472
Epoch 3/15
540/540 - 5s - 10ms/step - accuracy: 0.9629 - loss: 0.1230 - val_accuracy: 0.9647 - val_loss: 0.1332
Epoch 4/15
540/540 - 4s - 8ms/step - accuracy: 0.9684 - loss: 0.1030 - val_accuracy: 0.9717 - val_loss: 0.1026
Epoch 5/15
540/540 - 6s - 11ms/step - accuracy: 0.9744 - loss: 0.0846 - val_accuracy: 0.9740 - val_loss: 0.0943
Epoch 6/15
540/540 - 6s - 11ms/step - accuracy: 0.9766 - loss: 0.0751 - val_accuracy: 0.9732 - val_loss: 0.0859
Epoch 7/15
540/540 - 8s - 15ms/step - accuracy: 0.9804 - loss: 0.0638 - val_accuracy: 0.9803 - val_loss: 0.0708
Epoch 8/15
540/540 - 6s - 11ms/step - accuracy: 0.9825 - loss: 0.0575 - val_accuracy: 0.9817 - val_loss: 0.0627
Epoch 9/15
540/540 - 6s - 11ms/step - accuracy: 0.9832 - loss: 0.0516 - val_accuracy: 0.9817 - val_loss: 0.0636
Epo

## Testing

In [9]:
test_loss,test_accuracy = model.evaluate(test_data)

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.9726 - loss: 0.1017


In [10]:
print(f"Test Loss: {test_loss*100:.2f} %   Test Accuracy: {test_accuracy*100:.2f} % ")

Test Loss: 10.17 %   Test Accuracy: 97.26 % 
